# 🎓 Classification d'Ontologies - Démonstration Interactive

Ce notebook démontre le processus de classification d'ontologies en utilisant un modèle de Machine Learning entraîné.

## 📋 Étapes

1. Extraction des classes des ontologies A et B
2. Entraînement d'un modèle de classification sur A
3. Classification des classes de B
4. Visualisation des résultats

In [ ]:
# Installation des dépendances (si nécessaire)
# !pip install -q owlready2 sentence-transformers scikit-learn pandas matplotlib seaborn

In [ ]:
# Imports
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

# Ajouter le dossier Pipeline au path
sys.path.insert(0, str(Path.cwd() / "Pipeline"))

from extract_classes import OntologyClassExtractor
from classify_ontologies import OntologyClassifier

# Configuration pour les graphiques
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Imports réussis")

## 1️⃣ Extraction des Classes

In [ ]:
# Chemins des ontologies
data_dir = Path.cwd() / "Data"
onto_a_path = data_dir / "ontologie_animaux_A.owl"
onto_b_path = data_dir / "ontologie_animaux_B.owl"

print(f"📂 Ontologie A: {onto_a_path.name}")
print(f"📂 Ontologie B: {onto_b_path.name}")

In [ ]:
# Extraction ontologie A
print("Extraction de l'ontologie A...")
extractor_a = OntologyClassExtractor(str(onto_a_path))
classes_a = extractor_a.extract_classes()

print(f"✓ {len(classes_a)} classes extraites de l'ontologie A")
print(f"\nExemples:")
for cls in classes_a[:5]:
    print(f"  - {cls['label']}: {len(cls['synonyms'])} synonymes, {len(cls['comments'])} commentaires")

In [ ]:
# Extraction ontologie B
print("Extraction de l'ontologie B...")
extractor_b = OntologyClassExtractor(str(onto_b_path))
classes_b = extractor_b.extract_classes()

print(f"✓ {len(classes_b)} classes extraites de l'ontologie B")
print(f"\nExemples:")
for cls in classes_b[:5]:
    print(f"  - {cls['label']}: {len(cls['synonyms'])} synonymes, {len(cls['comments'])} commentaires")

## 2️⃣ Entraînement du Modèle

In [ ]:
# Sauvegarder temporairement les classes
temp_dir = Path.cwd() / "temp_notebook"
temp_dir.mkdir(exist_ok=True)

classes_a_path = temp_dir / "classes_a.json"
classes_b_path = temp_dir / "classes_b.json"

with open(classes_a_path, 'w') as f:
    json.dump(classes_a, f)

with open(classes_b_path, 'w') as f:
    json.dump(classes_b, f)

print("✓ Classes sauvegardées temporairement")

In [ ]:
# Créer le classificateur
print("Création du classificateur...")
classifier = OntologyClassifier(
    embedding_model='all-MiniLM-L6-v2',
    classifier_type='random_forest'
)

# Charger les classes
classifier.load_classes(str(classes_a_path), str(classes_b_path))

In [ ]:
# Entraîner le modèle
print("Entraînement du modèle (avec augmentation de données)...\n")
classifier.train_model(use_augmentation=True)

print("\n✅ Modèle entraîné avec succès!")

## 3️⃣ Classification

In [ ]:
# Classifier les classes de B
print("Classification des classes de B sur A...\n")
mappings = classifier.classify(top_k=5)

print(f"\n✅ {len(mappings)} classes classifiées")

## 4️⃣ Analyse des Résultats

In [ ]:
# Créer un DataFrame pour l'analyse
results = []
for mapping in mappings:
    best = mapping['best_match']
    results.append({
        'Classe B': mapping['class_b_label'],
        'Meilleure Correspondance': best['class_a_label'],
        'Probabilité': best['probability'],
        'Probabilité (%)': best['probability'] * 100
    })

df_results = pd.DataFrame(results)
df_results = df_results.sort_values('Probabilité', ascending=False)

# Afficher les 10 meilleurs résultats
print("\n📊 Top 10 Meilleurs Mappings:\n")
print(df_results.head(10).to_string(index=False))

In [ ]:
# Statistiques générales
print("\n📈 Statistiques Générales:\n")
print(f"Probabilité moyenne: {df_results['Probabilité'].mean():.4f}")
print(f"Probabilité médiane: {df_results['Probabilité'].median():.4f}")
print(f"Écart-type: {df_results['Probabilité'].std():.4f}")
print(f"Min: {df_results['Probabilité'].min():.4f}")
print(f"Max: {df_results['Probabilité'].max():.4f}")

# Niveaux de confiance
high_conf = (df_results['Probabilité'] >= 0.7).sum()
medium_conf = ((df_results['Probabilité'] >= 0.4) & (df_results['Probabilité'] < 0.7)).sum()
low_conf = (df_results['Probabilité'] < 0.4).sum()

print(f"\n🎯 Niveaux de Confiance:")
print(f"  Haute (≥0.7): {high_conf} ({high_conf/len(df_results)*100:.1f}%)")
print(f"  Moyenne (0.4-0.7): {medium_conf} ({medium_conf/len(df_results)*100:.1f}%)")
print(f"  Basse (<0.4): {low_conf} ({low_conf/len(df_results)*100:.1f}%)")

## 5️⃣ Visualisations

In [ ]:
# Distribution des probabilités
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogramme
axes[0].hist(df_results['Probabilité'], bins=20, edgecolor='black', alpha=0.7)
axes[0].axvline(df_results['Probabilité'].mean(), color='red', linestyle='--', label='Moyenne')
axes[0].axvline(df_results['Probabilité'].median(), color='green', linestyle='--', label='Médiane')
axes[0].set_xlabel('Probabilité')
axes[0].set_ylabel('Fréquence')
axes[0].set_title('Distribution des Probabilités de Classification')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot
axes[1].boxplot(df_results['Probabilité'], vert=True)
axes[1].set_ylabel('Probabilité')
axes[1].set_title('Box Plot des Probabilités')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Niveaux de confiance (pie chart)
fig, ax = plt.subplots(figsize=(8, 8))

sizes = [high_conf, medium_conf, low_conf]
labels = [f'Haute\n({high_conf})', f'Moyenne\n({medium_conf})', f'Basse\n({low_conf})']
colors = ['#2ecc71', '#f39c12', '#e74c3c']
explode = (0.1, 0, 0)

ax.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%',
       shadow=True, startangle=90, textprops={'fontsize': 12, 'weight': 'bold'})
ax.set_title('Répartition des Niveaux de Confiance', fontsize=14, weight='bold')

plt.show()

In [ ]:
# Top 15 mappings
fig, ax = plt.subplots(figsize=(12, 8))

top_15 = df_results.head(15).copy()
top_15['Label'] = top_15['Classe B'] + ' → ' + top_15['Meilleure Correspondance']

colors_bar = ['#2ecc71' if p >= 0.7 else '#f39c12' if p >= 0.4 else '#e74c3c' 
              for p in top_15['Probabilité']]

ax.barh(range(len(top_15)), top_15['Probabilité (%)'], color=colors_bar, edgecolor='black')
ax.set_yticks(range(len(top_15)))
ax.set_yticklabels(top_15['Label'], fontsize=9)
ax.set_xlabel('Probabilité (%)', fontsize=11, weight='bold')
ax.set_title('Top 15 Mappings par Probabilité', fontsize=13, weight='bold')
ax.grid(True, axis='x', alpha=0.3)
ax.invert_yaxis()

# Ajouter les valeurs sur les barres
for i, v in enumerate(top_15['Probabilité (%)']):
    ax.text(v + 1, i, f'{v:.1f}%', va='center', fontsize=8)

plt.tight_layout()
plt.show()

## 6️⃣ Exemples Détaillés

In [ ]:
# Afficher quelques exemples détaillés
print("\n" + "="*80)
print("EXEMPLES DE CLASSIFICATIONS DÉTAILLÉES")
print("="*80)

for i, mapping in enumerate(mappings[:5], 1):
    print(f"\n{i}. Classe B: {mapping['class_b_label']}")
    print("-" * 80)
    print(f"Description: {mapping['class_b_description']}\n")
    print("Top 5 prédictions:")
    
    for j, match in enumerate(mapping['top_matches'], 1):
        emoji = "🥇" if j == 1 else "🥈" if j == 2 else "🥉" if j == 3 else "  "
        print(f"  {emoji} {j}. {match['class_a_label']:20s} → {match['probability']:.4f} ({match['probability']*100:.2f}%)")
    print()

## 7️⃣ Nettoyage

In [ ]:
# Nettoyer les fichiers temporaires
import shutil

if temp_dir.exists():
    shutil.rmtree(temp_dir)
    print("✓ Fichiers temporaires supprimés")

## 🎉 Conclusion

Ce notebook a démontré :
1. ✅ L'extraction de classes depuis des ontologies OWL
2. ✅ L'entraînement d'un modèle de classification supervisé
3. ✅ La classification avec probabilités calibrées
4. ✅ L'analyse et la visualisation des résultats

Pour exécuter le pipeline complet, utilisez :
```bash
./run_pipeline.sh
```